# 01 – First snapshot: time signal and spectrum

Phase 0 check: load one raw IMS snapshot and plot it in the time and frequency domain.

Requires the extracted dataset under `data/raw/` (see `data/README.md`). The notebook is committed
without outputs; it stops with a clear error if no data is found.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from bearing_monitor import io

DATA_ROOT = Path("../data/raw")
FIGURE_DIR = Path("../docs/figures")

test_dirs = io.find_test_dirs(DATA_ROOT)
if not test_dirs:
    raise FileNotFoundError(f"No snapshot files below {DATA_ROOT.resolve()} - see data/README.md")
test_dirs

## Pick a snapshot

`TEST` selects the channel layout from `io.TEST_CONFIGS` (1, 2 or 3) and `TEST_DIR` the matching
folder. Adjust both if the folder order above differs.

In [ ]:
TEST = 1
TEST_DIR = test_dirs[0]
BEARING = 1
SNAPSHOT_INDEX = 0  # 0 = first (healthy) snapshot, -1 = last one before the test stopped

snapshots = io.list_snapshots(TEST_DIR)
path = snapshots.iloc[SNAPSHOT_INDEX]
timestamp = snapshots.index[SNAPSHOT_INDEX]
snapshot = io.load_snapshot(path, n_channels=io.TEST_CONFIGS[TEST].n_channels)
print(f"{len(snapshots)} snapshots from {snapshots.index[0]} to {snapshots.index[-1]}")
print(f"loaded {path.name}: shape {snapshot.shape}")

In [ ]:
signal = io.bearing_signal(snapshot, test=TEST, bearing=BEARING)
fs = io.SAMPLING_RATE_HZ  # nominal 20 kHz, see the open point in data/README.md
time_s = [i / fs for i in range(signal.size)]
freqs, spectrum = io.amplitude_spectrum(signal, fs)

fig, (ax_time, ax_freq) = plt.subplots(2, 1, figsize=(11, 7))
ax_time.plot(time_s, signal, linewidth=0.4)
ax_time.set(xlabel="Time [s]", ylabel="Acceleration [raw units]",
            title=f"Test {TEST}, bearing {BEARING}, snapshot {timestamp}")
ax_freq.plot(freqs, spectrum, linewidth=0.5)
ax_freq.set(xlabel="Frequency [Hz]", ylabel="Amplitude [raw units]",
            title="Single-sided amplitude spectrum (mean removed)", xlim=(0, fs / 2))
for ax in (ax_time, ax_freq):
    ax.grid(alpha=0.3)
fig.tight_layout()

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(FIGURE_DIR / "first_snapshot.png", dpi=150)
plt.show()

## What to look for

- Shaft frequency: 2000 rpm = 33.3 Hz.
- Expected bearing fault frequencies (to be recomputed in phase 2): BPFO about 236 Hz, BPFI about 297 Hz.
- Compare `SNAPSHOT_INDEX = 0` with `-1` to see how the signal changes towards the end of the run.